# Lab type: review
# Course: ML201 — Applied Machine Learning
# Lesson: Building Production-Ready ML Pipelines
# Task: The code below is correct and working. Read each section, run it, then answer the judgment questions in the markdown cells below each block.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import hashlib
import sklearn
from datetime import datetime
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

np.random.seed(42)
n = 2000

monthly_spend = np.random.lognormal(mean=4.0, sigma=0.6, size=n)
account_age = np.random.exponential(24, n).clip(1, 120)  # months
support_tickets = np.random.poisson(1.5, n).clip(0, 15).astype(float)
region = np.random.choice(['north', 'south', 'east', 'west'], n, p=[0.3, 0.25, 0.25, 0.2])
plan_type = np.random.choice(['basic', 'standard', 'pro'], n, p=[0.4, 0.4, 0.2])

# ~20% churn rate
plan_enc = np.where(plan_type == 'basic', 0.3, np.where(plan_type == 'standard', 0.0, -0.4))
log_odds_churn = (
    -2.5
    - 0.003 * monthly_spend
    - 0.015 * account_age
    + 0.35 * support_tickets
    + plan_enc
    + 0.15 * (region == 'north').astype(float)
)
prob_churn = 1 / (1 + np.exp(-log_odds_churn))
churned = np.random.binomial(1, prob_churn)

# Introduce 5% missing values in monthly_spend and support_tickets
miss_spend_idx = np.random.choice(n, size=int(0.05 * n), replace=False)
miss_tickets_idx = np.random.choice(n, size=int(0.05 * n), replace=False)
monthly_spend[miss_spend_idx] = np.nan
support_tickets[miss_tickets_idx] = np.nan

df = pd.DataFrame({
    'monthly_spend': monthly_spend,
    'account_age': account_age,
    'support_tickets': support_tickets,
    'region': region,
    'plan_type': plan_type,
    'churned': churned
})

print(f"Dataset shape   : {df.shape}")
print(f"Churn rate      : {df['churned'].mean():.2%}")
print("\nMissing values:")
print(df.isnull().sum())

X = df.drop('churned', axis=1)
y = df['churned']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

## Part 1: Pipeline + ColumnTransformer

In [ ]:
numeric_features = ['monthly_spend', 'account_age', 'support_tickets']
categorical_features = ['region', 'plan_type']

numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42))
])

pipe.fit(X_train, y_train)

test_auc = roc_auc_score(y_test, pipe.predict_proba(X_test)[:, 1])
print(f"Test AUC: {test_auc:.3f}")
print(f"\nPipeline steps: {list(pipe.named_steps.keys())}")
print("\nColumnTransformer transformers:")
for name, trans, cols in pipe.named_steps['preprocessor'].transformers_:
    print(f"  '{name}': {type(trans).__name__} -> columns {cols}")

**Question 1:** The pipeline was fitted with `pipe.fit(X_train, y_train)`. If new data arrives at inference time with some missing `monthly_spend` values, will the pipeline handle them correctly? Trace through what each step does.

*(Write your answer here.)*

**Question 2:** A colleague suggests separating preprocessing from the pipeline: fit the imputer and scaler on `X_train` separately, then create the pipeline with just the model. What goes wrong at deployment when raw data arrives for inference?

*(Write your answer here.)*

## Part 2: Serialise the Full Pipeline

> **Security note — joblib/pickle:** `joblib.dump` and `joblib.load` use Python's pickle protocol, which can execute arbitrary code when deserialising a malicious file. The loads below are safe because the files are written and read within this notebook session — no external or user-supplied bytes are deserialised. **In production, only load `.pkl` files that your own training code produced, stored in an access-controlled location (e.g. an internal artefact store or S3 with restricted IAM policies). Never load a pickle file received from an untrusted source.**

In [ ]:
# --- Common mistake: saving only the model step ---
rf_model_only = pipe.named_steps['model']
joblib.dump(rf_model_only, '/tmp/model_only.pkl')
print("Saved model-only artifact: /tmp/model_only.pkl")

# Safe to load: written two lines above in this same session.
loaded_model_only = joblib.load('/tmp/model_only.pkl')
try:
    preds_bad = loaded_model_only.predict_proba(X_test)[:, 1]
    print("Prediction succeeded (unexpected)")
except Exception as e:
    print(f"\nError when predicting with model-only artifact:")
    print(f"  {type(e).__name__}: {e}")

In [ ]:
# --- Correct pattern: save the full pipeline ---
joblib.dump(pipe, '/tmp/full_pipeline.pkl')

# Safe to load: written one line above in this same session.
loaded_pipe = joblib.load('/tmp/full_pipeline.pkl')
preds = loaded_pipe.predict_proba(X_test)[:, 1]
loaded_auc = roc_auc_score(y_test, preds)
print(f"Pipeline loaded successfully. AUC: {loaded_auc:.3f}")

**Question 3:** The `model_only.pkl` artifact works fine when called with pre-processed numpy arrays. Describe a specific scenario in a production API where this will silently produce wrong predictions rather than raising an error.

*(Write your answer here.)*

**Question 4:** If you update the model 3 months later (refit on new data), what must you serialise? What does NOT need to be re-serialised?

*(Write your answer here.)*

## Part 3: Reproducibility

In [ ]:
# Practice 1: Check all stochastic components have random_state
print("=== random_state parameters in the pipeline ===")
all_params = pipe.get_params()
random_state_params = {k: v for k, v in all_params.items() if 'random_state' in k}
for k, v in random_state_params.items():
    print(f"  {k}: {v}")

In [ ]:
# Practice 2: Log metadata
dataset_hash = hashlib.md5(
    pd.util.hash_pandas_object(X_train, index=True).values
).hexdigest()

metadata = {
    'trained_at': datetime.utcnow().isoformat(),
    'n_train': len(X_train),
    'test_auc': round(test_auc, 4),
    'sklearn_version': sklearn.__version__,
    'dataset_hash': dataset_hash
}

print("=== Training metadata ===")
for k, v in metadata.items():
    print(f"  {k}: {v}")

In [ ]:
# Practice 3: Pin requirements (key library versions)
import numpy
print("=== Key library versions ===")
print(f"  scikit-learn : {sklearn.__version__}")
print(f"  numpy        : {numpy.__version__}")
print(f"  pandas       : {pd.__version__}")
print("\nAdd these to requirements.txt or pyproject.toml to reproduce this environment.")

**Question 5:** If scikit-learn upgrades from 1.4 to 1.5 and the default value of a hyperparameter changes, your saved `pipeline.pkl` will still use the 1.4 default at load time. But what breaks when you try to retrain on new data using the same code without pinned versions?

*(Write your answer here.)*

**Question 6:** The `dataset_hash` is computed from `X_train`. What scenario would cause this hash to be the same even though the model's performance has changed significantly?

*(Write your answer here.)*

## Part 4: Pipeline Test

In [ ]:
def test_pipeline():
    # Safe to load: /tmp/full_pipeline.pkl was written by this notebook's Part 2 cell.
    loaded = joblib.load('/tmp/full_pipeline.pkl')
    test_row = pd.DataFrame([{
        'monthly_spend': 75.0,
        'account_age': 18,
        'support_tickets': 3,
        'region': 'west',
        'plan_type': 'standard'
    }])
    prob = loaded.predict_proba(test_row)[0, 1]
    assert 0.0 <= prob <= 1.0, f"Probability out of range: {prob}"
    print(f"Test passed. Predicted churn probability: {prob:.3f}")

test_pipeline()

**Question 7:** This test uses a hard-coded row with specific values. What three things does it verify? What would it NOT catch?

*(Write your answer here.)*

**Question 8:** In a CI/CD pipeline, where should this test run — before or after deployment? What events should trigger a re-run?

*(Write your answer here.)*

## Summary

Answer these final check questions in one sentence each:

1. Name two components that belong inside the sklearn `Pipeline` and one component that should NOT be inside it (and why).

2. You serialised the full pipeline, but a colleague re-fits only the model step and replaces `pipe.named_steps['model']` before saving. What can go wrong when the updated artifact is loaded in production?

3. What are the three elements of the reproducibility triad (what to log, what to pin, what to fix), and which one do developers most commonly skip?

4. A hard-coded "smoke test" on one row passes in CI. Give one example of a real-world input that would expose a bug the smoke test does not cover.